In [0]:
dbutils.widgets.dropdown("environment", "prod", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
from pyspark.sql.functions import col, to_date

config = load_config_from_widget()
bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.orders_raw"

df_clean = (spark.table(bronze_table)
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))
    .filter(col("quantity") > 0)
    .filter(col("total_amount") > 0)
)

silver_schema = config['schemas']['silver']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{silver_schema}")

silver_table = f"{config['catalog']}.{silver_schema}.orders_clean"
df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"✅ {silver_table}: {spark.table(silver_table).count()} records")